In [ ]:
"""Canonical bootstrap for revised_ark_bridge notebooks.

On a fresh Colab runtime this installs dependencies and clones the repo.
If you already have the repo checked out locally (e.g. running in Jupyter),
it detects it, adds it to sys.path, and skips all installation.
"""

import sys, os, subprocess, importlib, zipfile, shutil

repo_name = "neuro-analog"
repo_path = None

# ── FAST PATH: already inside a local checkout? ──
# Walk up from cwd looking for neuro-analog/
cwd = os.getcwd()
parts = cwd.replace("\\", "/").split("/")
for i in range(len(parts), 0, -1):
    candidate = "/".join(parts[:i])
    if candidate.endswith(repo_name) and os.path.isdir(os.path.join(candidate, "neuro_analog")):
        repo_path = candidate
        break

# Validate and short-circuit if we already have the repo locally
if repo_path is not None:
    if not os.path.isdir(os.path.join(repo_path, "neuro_analog", "revised_ark_bridge")):
        print(f"WARNING: Found repo at {repo_path} but missing revised_ark_bridge. Ignoring.")
        repo_path = None
    else:
        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)
        ckpt_root = os.path.join(repo_path, "experiments", "cross_arch_tolerance", "checkpoints")
        if not os.path.isdir(ckpt_root):
            print(f"WARNING: Checkpoints directory not found: {ckpt_root}. Proceeding with full setup...")
            repo_path = None
        else:
            # Verify Ark is reachable before short-circuiting
            try:
                import ark
            except ImportError:
                print(f"WARNING: repo found at {repo_path} but Ark is not importable.")
                print("Install Ark locally (e.g. pip install -e /path/to/Ark) and re-run.")
                raise SystemExit(1)
            print(f"Local repo detected at {repo_path} — skipping all installs.")
            print("Checkpoints path:", ckpt_root)
            print("\nSUCCESS! Setup complete.")
            raise SystemExit(0)

# ── SLOW PATH: fresh runtime / missing deps ──
# Force-upgrade jax + torch atomically so pip resolves a CUDA-compatible pair
subprocess.run(["pip", "install", "-q", "--upgrade", "jax[cuda12]", "torch"], check=False, capture_output=True, text=True)

# 1. Walk up from cwd looking for neuro-analog/
cwd = os.getcwd()
parts = cwd.replace("\\", "/").split("/")
for i in range(len(parts), 0, -1):
    candidate = "/".join(parts[:i])
    if candidate.endswith(repo_name) and os.path.isdir(os.path.join(candidate, "neuro_analog")):
        repo_path = candidate
        break

# 2. Check known Colab paths
if repo_path is None:
    for colab_path in [f"/content/{repo_name}", f"/content/real_new_content/{repo_name}"]:
        if os.path.isdir(os.path.join(colab_path, "neuro_analog")):
            repo_path = colab_path
            break

# 3. Check for uploaded zip files
if repo_path is None:
    for zip_dir in ["/content", "/content/real_new_content"]:
        zip_path = os.path.join(zip_dir, f"{repo_name}.zip")
        if os.path.exists(zip_path):
            extract_to = os.path.join(zip_dir, repo_name)
            if os.path.exists(extract_to):
                shutil.rmtree(extract_to)
            os.makedirs(extract_to, exist_ok=True)
            if not os.path.isfile(zip_path):
                print(f"Skipping {zip_path}: not a file (may be a directory).")
                continue
            file_size = os.path.getsize(zip_path)
            if file_size == 0:
                print(f"Skipping {zip_path}: file is empty.")
                continue
            print(f"Found zip at {zip_path} ({file_size} bytes). Extracting...")
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    for info in z.infolist():
                        # Windows backslashes in zip filenames must become Linux forward slashes
                        info.filename = info.filename.replace("\\", "/")
                        z.extract(info, extract_to)
            except zipfile.BadZipFile as e:
                print(f"Bad zip file at {zip_path}: {e}. Trying next location or git clone...")
                continue
            # Handle flat extraction (zip contains neuro_analog/ directly)
            if not os.path.isdir(os.path.join(extract_to, "neuro_analog")):
                flat_items = [d for d in os.listdir(extract_to) if os.path.isdir(os.path.join(extract_to, d))]
                if len(flat_items) == 1:
                    nested = os.path.join(extract_to, flat_items[0])
                    for item in os.listdir(nested):
                        shutil.move(os.path.join(nested, item), os.path.join(extract_to, item))
                    shutil.rmtree(nested)
            repo_path = extract_to
            print(f"Extracted zip to {repo_path}")
            break

# 4. Fallback: git clone
if repo_path is None:
    clone_target = f"/content/{repo_name}"
    print("Repository not found. Cloning from GitHub...")
    subprocess.run(
        ["git", "clone", "https://github.com/apumutyala/neuro-analog.git", clone_target],
        check=True, capture_output=True, text=True
    )
    repo_path = clone_target
    print(f"Cloned to {repo_path}")
else:
    print(f"Found existing repo at: {repo_path} (will validate and install missing deps)")

# Validate repo structure before accepting it
if repo_path is not None and not os.path.isdir(os.path.join(repo_path, "neuro_analog", "revised_ark_bridge")):
    print(f"WARNING: Found repo at {repo_path} but missing neuro_analog/revised_ark_bridge. Ignoring.")
    repo_path = None

# Add repo root to path
if repo_path is not None and repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# --- Install system dependencies for Ark ---
try:
    import graphviz
except ImportError:
    print("Installing graphviz system package...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz", "libgraphviz-dev"],
                   check=False, capture_output=True, text=True)

try:
    import z3
except ImportError:
    print("Installing z3-solver...")
    subprocess.run(["pip", "install", "-q", "z3-solver"],
                   check=True, capture_output=True, text=True)

# --- Install torch FIRST (Ark README says torch before jax due to cudnn) ---
try:
    import torch
    print("torch already available.")
except ImportError:
    print("Installing torch (must come before jax)...")
    subprocess.run(["pip", "install", "-q", "torch"],
                   check=True, capture_output=True, text=True)

# --- Install JAX + other Python dependencies ---
required_pkgs = ["jax", "jaxlib", "flax", "diffrax", "equinox", "matplotlib", "torchdiffeq", "scipy", "pysmt", "palettable"]
missing = []
for pkg in required_pkgs:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing packages: {missing}")
    if "jax" in missing or "jaxlib" in missing:
        subprocess.run(["pip", "install", "-q", "jax[cuda12]"],
                       check=True, capture_output=True, text=True)
        missing = [p for p in missing if p not in ("jax", "jaxlib")]
    if missing:
        subprocess.run(["pip", "install", "-q"] + missing,
                       check=True, capture_output=True, text=True)
    print("Installation complete.")
else:
    print("All required packages already installed.")

# --- Clone and install Ark ---
try:
    import ark
    print("Ark already available.")
except ImportError:
    print("Ark not found. Cloning from WangYuNeng/Ark...")
    ark_clone_target = "/content/Ark"
    if not os.path.exists(ark_clone_target):
        subprocess.run(
            ["git", "clone", "https://github.com/WangYuNeng/Ark.git", ark_clone_target],
            check=True, capture_output=True, text=True
        )
    print("Installing Ark from source...")
    subprocess.run(
        ["pip", "install", "-q", "-e", "."],
        cwd=ark_clone_target, check=True, capture_output=True, text=True
    )
    if ark_clone_target not in sys.path:
        sys.path.insert(0, ark_clone_target)
    print("Ark installed.")

# --- Set checkpoint path ---
ckpt_root = os.path.join(repo_path, "experiments", "cross_arch_tolerance", "checkpoints")
assert os.path.isdir(ckpt_root), f"Checkpoints directory not found: {ckpt_root}"
print("Checkpoints path:", ckpt_root)

# --- Final imports ---
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import diffrax
from ark.optimization.base_module import TimeInfo

print("JAX devices:", jax.devices())
print("\nSUCCESS! Setup complete.")


# Notebook 3 — Under the Hood: PyTorch -> Ark Compilation

Notebooks 1 and 2 showed the *behavior* of six analog circuits. This notebook shows **how the bridge is actually programmed** -- the real PyTorch -> Ark -> `BaseAnalogCkt` path -- by displaying the live source of each step and proving the compiler implements exactly the physics we specify.

**Two layers to keep straight:**
- **The CDG spec** (`core/paradigms.py`) -- a tiny *grammar* (node types, edge types, production rules) that *is* the ODE and maps one-to-one onto circuit elements.
- **The `BaseAnalogCkt` runtime** (Ark) -- an `equinox` module whose `__call__` runs a `diffrax` solve over `ode_fn`, with JAX autodiff flowing through the solve to a flat trainable parameter vector.

`OptCompiler.compile` turns the grammar into a **subclass of `BaseAnalogCkt`**. The three architectures that don't fit the grammar (Neural ODE, Flow, Diffusion) are hand-written `BaseAnalogCkt` subclasses implementing the *same* four methods -- so they share the identical solver and gradient machinery.

## 1. The grammar (`core/paradigms.py`)

`additive_recurrent` returns a `CDGSpec`: two node types (`StateVar`, `OutUnit`), two edge types (`MapEdge`, `FlowEdge`), and **three production rules** that encode `dx/dt = -x + b + sum_j g_ij * phi(x_j)`. Mismatch is declared right here -- `FlowEdge.g` becomes `AttrDefMismatch(rstd=sigma)` when `mismatch_sigma > 0`.

In [ ]:
import inspect
from neuro_analog.revised_ark_bridge.core import paradigms
print(inspect.getsource(paradigms.additive_recurrent))

## 2. Weights -> graph (`core/compiler.py`)

`build_additive_cdg` instantiates the graph from the PyTorch weights: one trainable conductance per **nonzero** weight (`abs(w) < 1e-12` is skipped = dead-node elimination), each minted into the flat parameter vector by the `TrainableMgr`.

In [ ]:
from neuro_analog.revised_ark_bridge.core import compiler
print(inspect.getsource(compiler.build_additive_cdg))

## 3. Compile -> a real `BaseAnalogCkt` subclass

`compile_cdg` builds the CDG and calls `OptCompiler().compile(...)`, which **dynamically generates a subclass of `BaseAnalogCkt`** via `type(name, (BaseAnalogCkt,), {ode_fn, make_args, noise_fn, readout, ...})`. Below we display `compile_cdg`, then build the *real* DEQ circuit and confirm the returned object is a genuine `BaseAnalogCkt` subclass -- a runtime circuit, **not** a source-code string (the fatal flaw of the old `ark_bridge`).

In [ ]:
print(inspect.getsource(compiler.compile_cdg))

# Build the REAL DEQ circuit from the trained PyTorch checkpoint
from experiments.cross_arch_tolerance.models.deq import load_model as load_deq
from neuro_analog.revised_ark_bridge.cdg_native.deq import build_deq
from ark.optimization.base_module import BaseAnalogCkt
import numpy as np

deq_pt = load_deq(os.path.join(ckpt_root, "deq.pt"))
CktClass, mgr, b_eff, rho = build_deq(deq_pt, mismatch_sigma=0.0, vectorize=True)

W_z_np = deq_pt.W_z.weight.detach().cpu().numpy()
n_cond = int(np.sum(np.abs(W_z_np) >= 1e-12))

print()
print("Generated class           :", CktClass.__name__)
print("Is BaseAnalogCkt subclass :", BaseAnalogCkt in CktClass.__mro__)
print("MRO                       :", [c.__name__ for c in CktClass.__mro__])
print("Conductances wired (nonzero W_z entries):", n_cond, "+", W_z_np.shape[0], "biases")

## 4. The shared runtime -- `BaseAnalogCkt.__call__` (Ark)

This is the engine **every** circuit inherits -- compiled or hand-written. It runs `args = make_args(...)` (mismatch injected here), then a `diffrax` solve: `ODETerm(ode_fn)` for deterministic dynamics, or `MultiTerm(ODETerm, ControlTerm)` with a Brownian term when `is_stochastic=True`. JAX autodiff traces **through the solve** to the flat `a_trainable` -- that is where differentiable, mismatch-aware retraining comes from.

In [ ]:
print(inspect.getsource(BaseAnalogCkt.__call__))

## 5. The compiled circuit is a real ODE solver that reaches a fixed point

The production rules in §1 define the additive-recurrent RHS `dz/dt = -z + b + W·tanh(z)`. The compiler turns that into the `ode_fn` of a `BaseAnalogCkt` subclass and integrates it with diffrax. Below we run the compiled DEQ — it reproduces the Notebook-1 settling trajectory — and confirm it reaches a fixed point by checking the state stops changing near the end of the run (`||z(t_end) - z(t_end-8)|| ≈ 0`). The curves going flat *is* the fixed point. (We check settling from the trajectory itself; the compiled circuit is built with `vectorize=True`, so calling its `ode_fn` by hand would not match the vmapped internal call the solver makes.)

In [ ]:
import numpy as np
n_state = deq_pt.W_z.weight.shape[0]
ti = TimeInfo(t0=0.0, t1=80.0, dt0=0.1, saveat=jnp.linspace(0.0, 80.0, 200))
z0 = jnp.zeros(n_state)

ckt = CktClass(init_trainable=mgr.get_initial_vals(), is_stochastic=False, solver=diffrax.Tsit5())
out = np.array(ckt(ti, z0, switch=jnp.array([]), args_seed=0, noise_seed=0))
print("compiled trajectory shape:", out.shape, " final state norm:", float(np.linalg.norm(out[-1])))

# Fixed point = the trajectory stops changing. Compare the end to ~8 time-units earlier.
settle = float(np.linalg.norm(out[-1] - out[-20]))
print(f"||z(t_end) - z(t_end-8)|| = {settle:.3e}   vs   ||z*|| = {np.linalg.norm(out[-1]):.3e}")
print("=> the state has stopped moving -> the generated circuit relaxed to a fixed point")

fig, ax = plt.subplots(figsize=(7, 3.5))
for i in [0, 15, 31, 47, 63]:
    ax.plot(np.array(ti.saveat), out[:, i], lw=2, alpha=0.8, label=f"z_{i}")
ax.set_title("Compiled DEQ circuit relaxes to its fixed point (reproduces Notebook 1)")
ax.set_xlabel("t"); ax.set_ylabel("z_i"); ax.axhline(0, color="k", lw=0.5)
ax.legend(fontsize=7, loc="upper right"); plt.tight_layout(); plt.show()

## 6. The hand-written path -- same contract, no codegen

When the physics doesn't fit the static grammar, we subclass `BaseAnalogCkt` directly and implement the same four methods. `MLPFieldCkt` (Neural ODE / Flow) builds `[z, t]` and runs a 3-layer MLP inside `ode_fn`; `CLDCkt` (Diffusion) sets `is_stochastic=True` and puts thermal noise on the momentum `v` only. Note the `make_args` mismatch is the **same** multiplicative `w*(1+sigma*N)` the compiler auto-generates.

In [ ]:
from neuro_analog.revised_ark_bridge.plain_fallback.mlp_field import MLPFieldCkt
from neuro_analog.revised_ark_bridge.plain_fallback.diffusion import CLDCkt
print("===== MLPFieldCkt (Neural ODE / Flow) =====")
print(inspect.getsource(MLPFieldCkt))
print("\n===== CLDCkt.ode_fn + noise_fn (Diffusion) =====")
print(inspect.getsource(CLDCkt.ode_fn))
print(inspect.getsource(CLDCkt.noise_fn))

## 7. (optional) Differentiability proof -- mismatch-aware retraining works for the hand-written path too

One `jax.grad` of a scalar loss on a hand-written circuit's output yields finite gradients w.r.t. *every* conductance -- JAX differentiates through the whole `diffrax` solve. (The first call JIT-compiles; give it ~a minute.)

In [ ]:
import jax, equinox as eqx
from experiments.cross_arch_tolerance.models.ssm import load_model as load_ssm
from neuro_analog.revised_ark_bridge.cdg_native.ssm import build_ssm

try:
    ssm_pt = load_ssm(os.path.join(ckpt_root, "ssm.pt"))
    ckt_ssm, _, _ = build_ssm(ssm_pt, mismatch_sigma=0.0, force_plain=True)  # hand-written LinearSSMCkt
    ti_g = TimeInfo(t0=0.0, t1=1.0, dt0=0.05, saveat=jnp.linspace(0.0, 1.0, 5))
    h0 = jnp.ones(ckt_ssm._A_shape[0])

    def loss(a):
        c = eqx.tree_at(lambda m: m.a_trainable, ckt_ssm, a)
        out = c(ti_g, h0, switch=jnp.array([]), args_seed=0, noise_seed=0)
        return jnp.sum(out ** 2)

    g = jax.grad(loss)(ckt_ssm.a_trainable)
    print("a_trainable size :", int(ckt_ssm.a_trainable.shape[0]), "(flattened A)")
    print("grad all finite  :", bool(jnp.all(jnp.isfinite(g))))
    print("grad norm        :", float(jnp.linalg.norm(g)))
    print("=> jax.grad flows through the diffrax solve to every conductance (hand-written class)")
except Exception as e:
    print("differentiability demo skipped:", repr(e))

## Summary -- one contract, two ways to author it

| Provided by `BaseAnalogCkt` (shared by ALL six) | Auto-generated by `OptCompiler` (CDG path only) |
|---|---|
| diffrax solver (`ODETerm` / `MultiTerm`+`ControlTerm`) | `ode_fn` synthesized from production rules |
| autodiff / `jax.grad` through the solve | `make_args` mismatch injection (`w*(1+sigma*N)`) |
| flat `a_trainable` parameter vector | weight scaling/clipping (straight-through) |
| `make_args/ode_fn/noise_fn/readout` contract | `vectorize` (vmap), dead-node elimination |
| the `__call__` forward pass | formal grammar <-> Kirchhoff verification |

**The bridge in one line:** extract PyTorch weights -> *either* compile from a grammar *or* hand-write a `BaseAnalogCkt` -> one differentiable simulation contract for all six families. The opposite of the old `ark_bridge`, which emitted Python *source strings* and never actually called the compiler.